# Validation A — published Table 1 regression

This notebook runs the five YAML cases that compare the thermal/PV pipeline with Table 1 of Silva-Oelker and Jaramillo-Fernandez (2022). No local installation and no S4 compilation are required: the cases use the reduced spectra published with the original work and committed to this repository.

**Learning goals**

1. Execute a literature regression from YAML-backed inputs.
2. Compare calculated and published PV and temperature quantities.
3. Distinguish a conditional regression from a live-optics validation.

**Evidence limit:** the source spectra are already hemispherically reduced and do not preserve angle or polarization. The atmospheric term therefore uses the documented angle-independent fallback. This notebook does not test S4.

## 1. Prepare the temporary Colab runtime

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shlex
import subprocess
import sys

from IPython.display import Image, Markdown, display

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("Open this notebook in Google Colab before running setup.")

PROJECT_DIR = Path("/content/radcoolpv-py")

def run_command(args: list[str], cwd: Path | None = None, capture: bool = False):
    print("$", shlex.join(args))
    return subprocess.run(
        args, cwd=cwd, check=True, text=True,
        capture_output=capture,
    )

if not PROJECT_DIR.exists():
    run_command([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/gsilvaoelker/radcoolpv-py.git",
        str(PROJECT_DIR),
    ])

run_command([
    sys.executable, "-m", "pip", "install", "--quiet", "--editable", ".",
], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)

print("Repository:", PROJECT_DIR)

## 2. Run all five Table 1 cases

In [ ]:
result = run_command(
    [sys.executable, "validations/validation A/run_table1_validation.py"],
    cwd=PROJECT_DIR,
    capture=True,
)
print(result.stdout)

## 3. Read the comparison

The principal PV outputs ($J_{sc}$, $P_{mpp}$, $V_{oc}$) and equilibrium temperature are close to the published table under the stored-spectrum assumptions. Reflected power is the weakest observable, with errors reaching roughly 10%, so the result should be described as a **conditional regression**, not a complete reproduction.

In [ ]:
display(Image(filename=str(
    PROJECT_DIR / "docs/site/_static/validations/validation_a_table1_errors_preview.png"
)))

## 4. Exercise

Open `validations/validation A/table1_flat_sodalime.yaml`, change only `thermal.convection_coefficient`, save it under a new filename, and run `radcoolpv run` on the copy. Record the change in equilibrium temperature and maximum power. Do not overwrite the validation input: that would destroy the reference case.

**Reference:** G. Silva-Oelker and J. Jaramillo-Fernandez, *Optics Express* 30, 32965–32977 (2022), [doi:10.1364/OE.466335](https://doi.org/10.1364/OE.466335).